# 11_Hybrid_Forecasting

Hybrid forecasting based on historical coverage.

```
>= 8 years → Prophet
4–7 years → Linear Regression
< 4 years → Feature-Based Forecast
```


In [1]:

import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')


c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

skills = pd.read_csv('../Generated Datasets/master_skill_dataset_v8.csv')
history = pd.read_csv('../Generated Datasets/skill_demand_history_v2.csv')

skills['skill'] = skills['skill'].str.lower().str.strip()
history['skill'] = history['skill'].str.lower().str.strip()

print(skills.shape)
print(history.shape)


(114, 13)
(1201, 3)


In [3]:

skills['skill'] = skills['skill'].str.lower().str.strip()
history['skill'] = history['skill'].str.lower().str.strip()

alias_map = {
    'node':'node.js',
    'nodejs':'node.js',
    'js':'javascript',
    'postgres':'postgresql',
    'azure':'microsoft azure',
    'sql server':'microsoft sql server',
    'ms sql server':'microsoft sql server',
    'pytorch':'torch/pytorch',
    'torch':'torch/pytorch',
    'react.js':'react',
    'reactjs':'react'
}

skills['skill'] = skills['skill'].replace(alias_map)
history['skill'] = history['skill'].replace(alias_map)

coverage = (
    history.groupby('skill')['year']
    .nunique()
    .reset_index(name='history_length')
)

skills = skills.drop(columns=['history_length'], errors='ignore')

skills = skills.merge(
    coverage,
    on='skill',
    how='left'
)

skills['history_length'] = skills['history_length'].fillna(0)

print("Matched skills:", skills['history_length'].gt(0).sum())
print(skills['history_length'].describe())


Matched skills: 31
count    114.000000
mean       2.166667
std        3.821033
min        0.000000
25%        0.000000
50%        0.000000
75%        3.000000
max       12.000000
Name: history_length, dtype: float64


In [4]:

def prophet_forecast(skill_name):

    df = history[history['skill']==skill_name].copy()

    prophet_df = pd.DataFrame({
        'ds': pd.to_datetime(df['year'].astype(str)+'-01-01'),
        'y': df['demand']
    })

    model = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=False,
        daily_seasonality=False
    )

    model.fit(prophet_df)

    future = pd.DataFrame({
        'ds': pd.to_datetime([
            '2026-01-01',
            '2027-01-01',
            '2028-01-01'
        ])
    })

    forecast = model.predict(future)

    return forecast.iloc[-1]['yhat']


In [5]:

def linear_forecast(skill_name):

    df = history[history['skill']==skill_name].copy()

    X = df[['year']]
    y = df['demand']

    model = LinearRegression()
    model.fit(X,y)

    pred = model.predict([[2028]])[0]

    return pred


In [6]:

feature_cols = [
    'linkedin_demand',
    'future_interest',
    'growth_rate',
    'current_usage',
    'global_adoption_score'
]

def feature_based_forecast(row):

    score = (
        0.30*row['linkedin_demand'] +
        0.25*row['future_interest'] +
        0.25*row['growth_rate'] +
        0.10*row['current_usage'] +
        0.10*row['global_adoption_score']
    )

    return score


In [7]:

results = []

for _, row in skills.iterrows():

    skill = row['skill']
    years = row['history_length']

    try:

        if years >= 8:

            forecast = prophet_forecast(skill)
            method = 'Prophet'

        elif years >= 4:

            forecast = linear_forecast(skill)
            method = 'Linear'

        else:

            forecast = feature_based_forecast(row)
            method = 'Feature-Based'

    except:

        forecast = feature_based_forecast(row)
        method = 'Feature-Based'

    results.append([
        skill,
        years,
        method,
        forecast
    ])

forecast_df = pd.DataFrame(
    results,
    columns=[
        'skill',
        'history_length',
        'forecast_method',
        'forecast_2028'
    ]
)

forecast_df.head()


19:57:52 - cmdstanpy - INFO - Chain [1] start processing
19:57:52 - cmdstanpy - INFO - Chain [1] done processing
19:57:52 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
19:57:52 - cmdstanpy - INFO - Chain [1] start processing
19:57:52 - cmdstanpy - INFO - Chain [1] done processing
19:57:52 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
19:57:52 - cmdstanpy - INFO - Chain [1] start processing
19:57:52 - cmdstanpy - INFO - Chain [1] done processing
19:57:52 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
19:57:52 - cmdstanpy - INFO - Chain [1] start processing
19:57:52 - cmdstanpy - INFO - Chain [1] done processing
19:57:52 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
19:57:53 - cmdstanpy - INFO - Chain [1] start processing
19:57:53 - cmdstanpy - INFO - Chain [1] done processing
19:57:53 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
19:57:53 - cmdstanpy - INFO - Chain [1] start processing
19:57:53 - cmdstanpy - INFO - Chain [1] don

,skill,history_length,forecast_method,forecast_2028
0,data analysis,0.0,Feature-Based,0.707592
1,excel,0.0,Feature-Based,0.524854
2,quality assurance,0.0,Feature-Based,0.441636
3,microsoft excel,0.0,Feature-Based,0.421330
4,python,10.0,Feature-Based,0.514275


In [8]:

scaler = MinMaxScaler()

forecast_df['forecast_score'] = scaler.fit_transform(
    forecast_df[['forecast_2028']]
)

def classify(x):

    if x >= 0.75:
        return 'Explosive'
    elif x >= 0.50:
        return 'Growing'
    elif x >= 0.25:
        return 'Stable'
    else:
        return 'Declining'

forecast_df['forecast_class'] = (
    forecast_df['forecast_score']
    .apply(classify)
)

forecast_df.head()


,skill,history_length,forecast_method,forecast_2028,forecast_score,forecast_class
0,data analysis,0.0,Feature-Based,0.707592,0.000013,Declining
1,excel,0.0,Feature-Based,0.524854,0.000009,Declining
2,quality assurance,0.0,Feature-Based,0.441636,0.000007,Declining
3,microsoft excel,0.0,Feature-Based,0.421330,0.000007,Declining
4,python,10.0,Feature-Based,0.514275,0.000009,Declining


In [9]:

final_df = skills.merge(
    forecast_df[
        [
            'skill',
            'history_length',
            'forecast_method',
            'forecast_2028',
            'forecast_score',
            'forecast_class'
        ]
    ],
    on='skill',
    how='left'
)

print(final_df.shape)
final_df.head()


(116, 19)


,skill,sub_category,linkedin_demand,salary_premium,current_usage,future_interest,global_adoption_score,growth_rate,history_found,cluster,pc1,pc2,archetype,history_length_x,history_length_y,forecast_method,forecast_2028,forecast_score,forecast_class
0,data analysis,Data Analytics,1.000000,0.326209,0.625994,0.559777,0.634146,0.566533,1,3,3.282909,-5.959951,Future-Proof,0.0,0.0,Feature-Based,0.707592,0.000013,Declining
1,excel,Data Analytics,0.506295,0.067637,0.340903,0.535307,0.634146,0.566533,1,2,0.823164,-4.208818,Growing,0.0,0.0,Feature-Based,0.524854,0.000009,Declining
2,quality assurance,Software Engineering,0.442621,0.175857,0.275063,0.305183,0.634146,0.566533,1,2,-0.244958,-3.534908,Growing,0.0,0.0,Feature-Based,0.441636,0.000007,Declining
3,microsoft excel,Data Analytics,0.314443,0.067637,0.233033,0.394582,0.634146,0.566533,1,2,-0.466162,-3.203326,Growing,0.0,0.0,Feature-Based,0.421330,0.000007,Declining
4,python,Programming,0.307062,0.631282,0.876458,0.701599,0.951220,0.255955,1,3,5.283296,-1.543775,Future-Proof,10.0,10.0,Feature-Based,0.514275,0.000009,Declining


In [10]:

final_df.to_csv(
    '../Generated Datasets/master_skill_dataset_v9.csv',
    index=False
)

print('Saved Successfully')
print(final_df['forecast_method'].value_counts())
print(final_df['forecast_class'].value_counts())


Saved Successfully
forecast_method
Feature-Based    108
Linear             8
Name: count, dtype: int64
forecast_class
Declining    112
Stable         2
Explosive      1
Growing        1
Name: count, dtype: int64


In [11]:
print(history.shape)

coverage = (
    history.groupby('skill')['year']
    .nunique()
)

print(coverage.describe())

print(coverage[coverage >= 8].count())
print(coverage[coverage >= 4].count())

(1201, 3)
count    330.000000
mean       3.639394
std        2.811228
min        1.000000
25%        1.000000
50%        3.000000
75%        5.000000
max       12.000000
Name: year, dtype: float64
45
114


In [12]:
print(
    len(
        set(history['skill'])
        &
        set(skills['skill'])
    )
)

30
